In [1]:
# Bitonic Sort using CUDA in Python (Numba)

import numpy as np
from numba import cuda
import math
import time

# CUDA Kernel for Bitonic Sort
@cuda.jit
def bitonic_sort_step(dev_values, j, k):
    i = cuda.grid(1)

    ixj = i ^ j

    if ixj > i:
        # Ascending order
        if (i & k) == 0:
            if dev_values[i] > dev_values[ixj]:
                temp = dev_values[i]
                dev_values[i] = dev_values[ixj]
                dev_values[ixj] = temp

        # Descending order
        if (i & k) != 0:
            if dev_values[i] < dev_values[ixj]:
                temp = dev_values[i]
                dev_values[i] = dev_values[ixj]
                dev_values[ixj] = temp


# Main Function
def bitonic_sort(arr):

    n = len(arr)

    # Copy data to GPU
    dev_arr = cuda.to_device(arr)

    threads_per_block = 256
    blocks = (n + threads_per_block - 1) // threads_per_block

    # Bitonic Sort
    k = 2
    while k <= n:

        j = k // 2

        while j > 0:
            bitonic_sort_step[blocks, threads_per_block](dev_arr, j, k)

            cuda.synchronize()

            j = j // 2

        k = k * 2

    # Copy sorted data back to CPU
    sorted_arr = dev_arr.copy_to_host()

    return sorted_arr


# Driver Code
if __name__ == "__main__":

    # Size must be power of 2
    N = 16

    arr = np.random.randint(0, 100, N).astype(np.int32)

    print("Original Array:")
    print(arr)

    start = time.time()

    sorted_arr = bitonic_sort(arr)

    end = time.time()

    print("\nSorted Array:")
    print(sorted_arr)

    print("\nExecution Time:", end - start, "seconds")

Original Array:
[70 63 56 71 31 89 38 43 92 73 44 98 72 96 43 27]


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))



Sorted Array:
[27 31 38 43 43 44 56 63 70 71 72 73 89 92 96 98]

Execution Time: 2.3781802654266357 seconds
